In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Anand_Vihar_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,353.0,189.0,268.0,NaN,237.0,316.0,NaN,113.0,269.0,295.0,389.0,306.0
1,2,366.0,223.0,116.0,157.0,188.0,212.0,201.0,NaN,170.0,400.0,401.0,311.0
2,3,360.0,215.0,116.0,201.0,279.0,NaN,135.0,NaN,98.0,380.0,436.0,311.0
3,4,427.0,321.0,140.0,245.0,297.0,262.0,NaN,98.0,147.0,400.0,429.0,211.0
4,5,382.0,203.0,145.0,243.0,380.0,392.0,NaN,89.0,111.0,390.0,428.0,190.0
5,6,372.0,185.0,130.0,203.0,278.0,221.0,108.0,99.0,140.0,383.0,NaN,248.0
6,7,372.0,242.0,224.0,213.0,404.0,268.0,83.0,119.0,91.0,329.0,434.0,310.0
7,8,406.0,170.0,162.0,249.0,266.0,291.0,87.0,68.0,162.0,375.0,413.0,374.0
8,9,415.0,187.0,137.0,308.0,208.0,186.0,106.0,73.0,242.0,371.0,377.0,219.0
9,10,320.0,366.0,188.0,279.0,NaN,186.0,167.0,84.0,125.0,314.0,353.0,264.0


In [4]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    34 non-null     float64
 2   February   33 non-null     float64
 3   March      22 non-null     float64
 4   April      24 non-null     float64
 5   May        33 non-null     float64
 6   June       25 non-null     float64
 7   July       33 non-null     float64
 8   August     34 non-null     float64
 9   September  28 non-null     float64
 10  October    35 non-null     float64
 11  November   33 non-null     float64
 12  December   36 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


(41, 13)

In [ ]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

In [ ]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [ ]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [ ]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready